In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from app.ml.data.generate_sales_data import SalesDataGenerator
from app.ml.forecasting.xgboost_forecaster import XGBoostForecaster
from app.ml.analysis.causal_analyzer import CausalAnalyzer
import shap

In [ ]:
# Synthetic data generation
generator = SalesDataGenerator(random_seed=42)
df = generator.generate(days=540)
df['ds'] = pd.to_datetime(df['date'])
dfc = df[['ds', 'sales']].rename(columns={'sales': 'y'})
dfc = dfc.sort_values('ds').reset_index(drop=True)
dfc.head()

In [ ]:
# Prepare XGBoost features
xgb_forecaster = XGBoostForecaster(model_dir='models')
xgb_data = xgb_forecaster.create_features(dfc.rename(columns={'ds':'ds','y':'y'}))
X = xgb_data.drop(columns=['ds', 'y'])
y = xgb_data['y']
X_train, X_test = X.iloc[:-90], X.iloc[-90:]
y_train, y_test = y.iloc[:-90], y.iloc[-90:]
xgb_forecaster.train(X_train, y_train, product_id='prod_001')

In [ ]:
# Causal SHAP analysis
causal = CausalAnalyzer()
analysis = causal.analyze_drivers(xgb_forecaster.model, X_test, top_n=5)
analysis

In [ ]:
# SHAP summary plot
explainer = shap.Explainer(xgb_forecaster.model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test, plot_type='bar')

In [ ]:
# Interaction effects for top two features
top_feats = [f['feature'] for f in analysis['top_drivers'][:2]]
if len(top_feats) >= 2:
    shap.dependence_plot(top_feats[0], shap_values.values, X_test, interaction_index=top_feats[1])

In [ ]:
# Explain sales change
prev_sales = 10000
curr_sales = 11600
factors = {'weekend_effect': 0.4, 'holiday': 0.3, 'weather': 0.2, 'promotion': 0.1}
print(causal.explain_change(curr_sales, prev_sales, factors))